# Pre-flight for the RSNA ConvNeXt run

`RSNA_External_Validation_Prep_Colab.ipynb` was written against Google Drive paths. On
Kaggle none of them exist, and Kaggle has also changed where competition data mounts.
This notebook finds every input by name, checks it, and prints a **config block to paste
into that notebook's cell 2**.

It reads nothing but file headers. No image is decoded, and the holdout is not opened.

Section 0 rebuilds the split file first, so this works in a fresh session.

**Save Version -> Save & Run All (Commit)** before closing, or `/kaggle/working`
is wiped and the split file disappears again.

## 0. Rebuild the split file

`/kaggle/working` is wiped when a session ends, so the `rsna_dev_holdout_split.json` from
the earlier run is gone. This rebuilds it from `rsna_split_manifest.csv` in your attached
Sprint5 output.

**This is a key rename, not a new split.** Sprint 5 wrote `split = 'dev' | 'holdout'`; the
prep notebook reads `'development' | 'holdout'`. No patient is moved, added or removed, and
the counts are checked against what Sprint 5 recorded. If they disagree the cell says so
loudly — a different split means the sealed holdout is not the one that was sealed.

No image is opened. The holdout is not read.

In [1]:
# Rebuilds rsna_dev_holdout_split.json into THIS session's /kaggle/working.
# It is a key rename, not a new split: no patient moves, and the counts are checked
# against what the Sprint 5 calibration run recorded.

import os, json, hashlib, glob
import pandas as pd

EXPECTED_DEV, EXPECTED_HOLDOUT = 6214, 2663
OUT_PATH = '/kaggle/working/rsna_dev_holdout_split.json'

src = None
for p in glob.glob('/kaggle/input/**/rsna_split_manifest.csv', recursive=True):
    src = os.path.realpath(p); break
if src is None:
    print('rsna_split_manifest.csv not found. Every CSV the kernel can see:')
    for q in sorted({os.path.realpath(p) for p in
                     glob.glob('/kaggle/input/**/*.csv', recursive=True)})[:60]:
        print('   ', q)
    raise SystemExit(
        'The manifest is the file the Sprint 5 run used to prove the holdout was excluded. '
        'Attach the Sprint5 notebook output. Do not derive a new split - that would move '
        'protected patients into the evaluated set.')

print('source:', src)
man = pd.read_csv(src)
man['patient_id'] = man['patient_id'].astype(str)
man['split'] = man['split'].astype(str).str.strip().str.lower()
print(man['split'].value_counts().to_string())

dev = sorted(set(man.loc[man.split == 'dev', 'patient_id']))
hold = sorted(set(man.loc[man.split == 'holdout', 'patient_id']))

overlap = set(dev) & set(hold)
if overlap:
    raise SystemExit('%d patients in BOTH partitions, e.g. %s - manifest is inconsistent'
                     % (len(overlap), sorted(overlap)[:5]))

ok = (len(dev) == EXPECTED_DEV) and (len(hold) == EXPECTED_HOLDOUT)
print()
print('development %6d  (Sprint 5 recorded %d)' % (len(dev), EXPECTED_DEV))
print('holdout     %6d  (project record %d)   SEALED' % (len(hold), EXPECTED_HOLDOUT))
print('no patient in both partitions  [PASS]')
print('counts match Sprint 5 exactly  [%s]' % ('PASS' if ok else 'FAIL'))
if not ok:
    print('*** STOP. A different split means the holdout is not the one that was sealed.')

json.dump({'development': dev, 'holdout': hold,
           'provenance': {'derived_from': os.path.basename(src),
                          'source_sha256': hashlib.sha256(open(src,'rb').read()).hexdigest(),
                          'operation': 'key rename only (dev -> development)',
                          'counts_match_sprint5': bool(ok)}},
          open(OUT_PATH, 'w'), indent=2)
print('\nwrote', OUT_PATH)

back = json.load(open(OUT_PATH))
assert set(back['development']) == set(dev) and set(back['holdout']) == set(hold)
print('round trip: identical membership  [PASS]')
print('\nNow re-run section 1. And use Save & Run All (Commit) before you close the')
print('session, or /kaggle/working is wiped again.')


source: /kaggle/input/notebooks/pasindupahasara/sprint5/rsna_split_manifest.csv
split
dev        6214
holdout    2663

development   6214  (Sprint 5 recorded 6214)
holdout       2663  (project record 2663)   SEALED
no patient in both partitions  [PASS]
counts match Sprint 5 exactly  [PASS]

wrote /kaggle/working/rsna_dev_holdout_split.json
round trip: identical membership  [PASS]

Now re-run section 1. And use Save & Run All (Commit) before you close the
session, or /kaggle/working is wiped again.


## 1. Find everything

Four things are needed: the RSNA competition data, the split JSON, the two
`preprocessing.py` modules (one per model), and the raw Mammo-Bench collection for the
control. Each is located by name rather than by an assumed path.

In [2]:
import os, json, hashlib, glob

def walk_find(name, roots=('/kaggle/input', '/kaggle/working')):
    """Every file with this exact basename, deduped."""
    hits = set()
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dp, _d, fs in os.walk(r):
            if name in fs:
                hits.add(os.path.realpath(os.path.join(dp, name)))
    return sorted(hits)

def sha256(p):
    return hashlib.sha256(open(p, 'rb').read()).hexdigest()

FOUND = {}
MISSING = []

# ---- RSNA competition data: handle both the old and the new mount layout ----
rsna_csvs = [p for p in walk_find('train.csv') if 'rsna' in p.lower()]
if rsna_csvs:
    FOUND['rsna_root'] = os.path.dirname(rsna_csvs[0])
    print('RSNA root        ', FOUND['rsna_root'])
    imgs = os.path.join(FOUND['rsna_root'], 'train_images')
    print('  train_images   ', 'present' if os.path.isdir(imgs) else 'MISSING')
    if not os.path.isdir(imgs):
        MISSING.append('train_images under the RSNA root')
else:
    MISSING.append('RSNA train.csv')
    print('RSNA root         NOT FOUND')

# ---- the split file written by the conversion notebook ----
spl = walk_find('rsna_dev_holdout_split.json')
if spl:
    FOUND['split'] = spl[0]
    d = json.load(open(spl[0]))
    print('split file       ', spl[0])
    print('  development    ', len(d.get('development', [])), 'patients')
    print('  holdout        ', len(d.get('holdout', [])), 'patients (sealed)')
else:
    MISSING.append('rsna_dev_holdout_split.json - run the split conversion notebook first')
    print('split file        NOT FOUND')

# ---- the two preprocessing modules, told apart by their recorded target ----
print()
preps = walk_find('preprocessing.py')
print('preprocessing.py copies found:', len(preps))
for p in preps:
    txt = open(p, encoding='utf-8', errors='replace').read()
    ver = 'prep-v2' if '2048' in txt else ('prep-v1' if '760' in txt else '?')
    th = '2048x1024' if '2048' in txt else ('760x456' if '760' in txt else '?')
    print('   %-9s %-11s %s' % (ver, th, p))
    print('             sha256 %s' % sha256(p))
    FOUND.setdefault('prep_' + ver, p)
if 'prep_prep-v2' not in FOUND:
    MISSING.append("prep-v2 preprocessing.py (2048x1024) - the ConvNeXt one, "
                   "sha256 3c2259f535e34c6c...")
if 'prep_prep-v1' not in FOUND:
    print('   (prep-v1 not found - not needed, this run is ConvNeXt only)')

# ---- the bundle and its contract module, needed by the SCORING notebook ----
print()
for nm, key in (('convnextv2h_model_bundle.pth', 'bundle'),
                ('model_input_v3.py', 'contract')):
    h = walk_find(nm)
    if h:
        FOUND[key] = h[0]
        print('%-24s %s (%.2f GB)' % (nm, h[0], os.path.getsize(h[0]) / 1e9))
    else:
        MISSING.append(nm + ' - needed to score, attach the bundle dataset')
        print('%-24s NOT FOUND' % nm)

# ---- raw Mammo-Bench, for the control ----
print()
cands = []
for r in ('/kaggle/input',):
    if os.path.isdir(r):
        cands += glob.glob(os.path.join(r, '**', 'dataset.zip'), recursive=True)
        cands += [d for d in glob.glob(os.path.join(r, '**', 'metadata.csv'), recursive=True)]
cands = sorted(set(os.path.realpath(c) for c in cands))
if cands:
    FOUND['mammobench'] = [os.path.dirname(c) if c.endswith('metadata.csv') else c
                           for c in cands]
    print('Mammo-Bench raw candidates:')
    for c in FOUND['mammobench']:
        print('   ', c)
else:
    MISSING.append('raw Mammo-Bench (dataset.zip or metadata.csv) for the control')
    print('Mammo-Bench raw   NOT FOUND')

print()
if MISSING:
    print('MISSING:')
    for m in MISSING:
        print('  -', m)
else:
    print('everything needed was found')

RSNA root         /kaggle/input/competitions/rsna-breast-cancer-detection
  train_images    present
split file        /kaggle/working/rsna_dev_holdout_split.json
  development     6214 patients
  holdout         2663 patients (sealed)

preprocessing.py copies found: 1
   prep-v2   2048x1024   /kaggle/input/datasets/pasindupahasara/qinterval-prep-v2/preprocessing.py
             sha256 3c2259f535e34c6cbffd1a3a42c63d8bdf1f48f7c8eb7653d9bc75d50673a160
   (prep-v1 not found - not needed, this run is ConvNeXt only)

convnextv2h_model_bundle.pth /kaggle/input/datasets/pasindupahasara/qinterval-prep-v2/convnextv2h_model_bundle.pth (2.63 GB)
model_input_v3.py        /kaggle/input/datasets/pasindupahasara/qinterval-prep-v2/model_input_v3.py (0.00 GB)

Mammo-Bench raw candidates:
    /kaggle/input/datasets/pasindupahasara/mammobench-test-control/mammobench_test
    /kaggle/input/datasets/pasindupahasara/mammobench-test-control

everything needed was found


## 2. One decision: `NEG_PER_POS`

The prep notebook defaults to **3**, with this reasoning in its own comment:

> *"12:1 → 3:1 is a 3.25x reduction in decode cost for 1.6 THOUSANDTHS of an AUC point.
> Under a time budget this is the cut to make."*

That is right **if the number you want is ConvNeXt's own external AUC**.

It is wrong if you want to put ConvNeXt next to the v12 result. Sprint 4 and the Sprint 5
calibration both ran at **12 per positive**, and the frozen operating point — sensitivity
0.2974 at 100 recalls per 1,000 — was computed on that cohort with per-site weights
reconstructed from it. A 3:1 run gives a different image set, so recall rate and PPV are
not comparable even after reweighting.

The notebook says as much: *"Raise this to 12 only if the run is being redone for strict
comparability with v13, and say so in the write-up."*

**Comparing against v12 is exactly that case.** Set 12, and say so.

In [3]:
# What the comparison costs, using the notebook's own table.
for npp, total, ci in ((12, 15054, 0.0178), (6, 8106, 0.0183),
                       (3, 4632, 0.0194), (1, 2316, 0.0229)):
    tag = '  <- Sprint 5 / v13 used this' if npp == 12 else (
          '  <- prep notebook default' if npp == 3 else '')
    print('  %2d:1   %6d images   CI half-width +/- %.4f%s' % (npp, total, ci, tag))
print()
print('Choose 12 for a like-for-like comparison with the frozen v12 operating point.')
print('Choose 3 only if you want ConvNeXt\'s standalone AUC and nothing else.')

NEG_PER_POS = 12          # <-- set this deliberately

  12:1    15054 images   CI half-width +/- 0.0178  <- Sprint 5 / v13 used this
   6:1     8106 images   CI half-width +/- 0.0183
   3:1     4632 images   CI half-width +/- 0.0194  <- prep notebook default
   1:1     2316 images   CI half-width +/- 0.0229

Choose 12 for a like-for-like comparison with the frozen v12 operating point.
Choose 3 only if you want ConvNeXt's standalone AUC and nothing else.


## 3. The config block

Copy everything this prints into cell 2 of `RSNA_External_Validation_Prep_Colab.ipynb`,
replacing the values it already has. Nothing else in that notebook changes.

In [4]:
if MISSING:
    raise SystemExit('Not every input was found - fix section 1 before pasting a config.\n'
                     + '\n'.join('  - ' + m for m in MISSING))

OUT_ROOT = '/kaggle/working/rsna_external/prep_v1'
mb = FOUND['mammobench']

L = []
A = L.append
A("# ---- discovered by the pre-flight notebook, %s ----" % __import__('datetime')
  .datetime.now().strftime('%Y-%m-%d'))
A("ENVIRONMENT      = 'kaggle'")
A("KAGGLE_RSNA_ROOT = %r" % FOUND['rsna_root'])
A("")
A("RSNA_PARTITION     = 'development'      # the holdout stays sealed")
A("RSNA_UNSEAL_TOKEN  = None")
A("RSNA_UNSEAL_REASON = None")
A("RSNA_SPLIT_FILE    = %r" % FOUND['split'])
A("")
A("NEG_PER_POS       = %d      # matches Sprint 5 / v13, for comparability" % NEG_PER_POS)
A("SAMPLE_SEED       = 42")
A("USE_ALL_NEGATIVES = False")
A("")
A("MODELS = {          # ConvNeXt only - mammofm removed, nothing to compare against")
for key, name, h, w, ver in (('prep_prep-v2', 'convnextv2h', 2048, 1024, 'prep-v2'),):
    p = FOUND[key]
    A("    %r: {" % name)
    A("        'prep_module': %r," % p)
    A("        'prep_sha256': %r," % sha256(p))
    A("        'expect_prep_version': %r," % ver)
    A("        'expect_h': %d, 'expect_w': %d," % (h, w))
    A("        'internal_manifest': %r," % os.path.join(os.path.dirname(p), 'manifest.csv'))
    A("        'internal_export_dir': %r," % os.path.dirname(p))
    A("    },")
A("}")
A("")
A("MAMMOBENCH_RAW_SOURCES = [")
for c in mb:
    A("    %r," % c)
A("]")
A("")
A("OUT_ROOT      = %r" % OUT_ROOT)
A("LOCAL_SCRATCH = '/kaggle/working/rsna_prep'")

print('\n'.join(L))
open('/kaggle/working/prep_config_block.py', 'w').write('\n'.join(L) + '\n')
print()
print('also saved to /kaggle/working/prep_config_block.py')

# ---- discovered by the pre-flight notebook, 2026-10-09 ----
ENVIRONMENT      = 'kaggle'
KAGGLE_RSNA_ROOT = '/kaggle/input/competitions/rsna-breast-cancer-detection'

RSNA_PARTITION     = 'development'      # the holdout stays sealed
RSNA_UNSEAL_TOKEN  = None
RSNA_UNSEAL_REASON = None
RSNA_SPLIT_FILE    = '/kaggle/working/rsna_dev_holdout_split.json'

NEG_PER_POS       = 12      # matches Sprint 5 / v13, for comparability
SAMPLE_SEED       = 42
USE_ALL_NEGATIVES = False

MODELS = {          # ConvNeXt only - mammofm removed, nothing to compare against
    'convnextv2h': {
        'prep_module': '/kaggle/input/datasets/pasindupahasara/qinterval-prep-v2/preprocessing.py',
        'prep_sha256': '3c2259f535e34c6cbffd1a3a42c63d8bdf1f48f7c8eb7653d9bc75d50673a160',
        'expect_prep_version': 'prep-v2',
        'expect_h': 2048, 'expect_w': 1024,
        'internal_manifest': '/kaggle/input/datasets/pasindupahasara/qinterval-prep-v2/manifest.csv',
        'internal_export_dir': '/kaggle/i

## 4. Two things to check by eye

The `prep_sha256` values above are what the files on Kaggle actually hash to. The prep
notebook verifies them against what each model bundle recorded — so if a hash here differs
from the bundle's, that is the check doing its job, not a nuisance. Do not paste a hash to
make an error go away.

In [5]:
EXPECT_PREP_V2 = '3c2259f535e34c6cbffd1a3a42c63d8bdf1f48f7c8eb7653d9bc75d50673a160'
EXPECT_CONTRACT = 'ec13e655ee8a34e658e771093532e04268276b5ecae851624947fc405727ed79'

got = sha256(FOUND['prep_prep-v2'])
print('prep-v2 preprocessing.py')
print('   this session :', got)
print('   bundle records:', EXPECT_PREP_V2)
print('   match        :', got == EXPECT_PREP_V2)
print()
if 'contract' in FOUND:
    gc = sha256(FOUND['contract'])
    print('model_input_v3.py')
    print('   this session :', gc)
    print('   bundle records:', EXPECT_CONTRACT)
    print('   match        :', gc == EXPECT_CONTRACT)
    print()
print('If a hash does not match, STOP. A different preprocessing.py means the images this')
print('run produces are not the ones that model was trained on, and the result would be')
print('measuring the mismatch rather than the model.')

prep-v2 preprocessing.py
   this session : 3c2259f535e34c6cbffd1a3a42c63d8bdf1f48f7c8eb7653d9bc75d50673a160
   bundle records: 3c2259f535e34c6cbffd1a3a42c63d8bdf1f48f7c8eb7653d9bc75d50673a160
   match        : True

model_input_v3.py
   this session : ec13e655ee8a34e658e771093532e04268276b5ecae851624947fc405727ed79
   bundle records: ec13e655ee8a34e658e771093532e04268276b5ecae851624947fc405727ed79
   match        : True

If a hash does not match, STOP. A different preprocessing.py means the images this
run produces are not the ones that model was trained on, and the result would be
measuring the mismatch rather than the model.


## 5. Then

1. paste the block from section 3 into cell 2 of the prep notebook
2. run it — section 6 times a short probe and projects the full pass **before** committing
   hours, so you find out early if it will not fit
3. `RESUME = True` is already set, so a timed-out session picks up where it stopped
4. **Save Version → Save & Run All (Commit)** or `/kaggle/working` is wiped

Then the scoring notebook, then the Sprint 5 calibration cells unchanged.

The holdout stays sealed throughout. It opens once, later, for one model, deliberately.